# Triton Inference Server Benchmark

Benchmark Triton-served models using `perf_analyzer`.

Modes:
- **Concurrency-range**: sweep concurrent request levels
- **Request-rate-range**: control incoming request rate
- **Poisson distribution**: simulate realistic traffic

Metrics: queue time, compute/inference time, throughput, latency percentiles

In [ ]:
import subprocess
import pandas as pd

TRITON_URL = "triton_server:8000"

MODELS_TO_BENCHMARK = [
    "distilbert_classifier_pt",
    "minilm_classifier_pt",
    "fasttext_classifier",
    "distilbert_classifier",  # ONNX
    "minilm_classifier",      # ONNX
    "tfidf_logreg_classifier", # ONNX
]

## Verify Triton is running

In [ ]:
import requests

resp = requests.get(f"http://{TRITON_URL}/v2/health/ready")
print(f"Triton health: {resp.status_code}")

resp = requests.get(f"http://{TRITON_URL}/v2/models/stats")
if resp.status_code == 200:
    stats = resp.json()
    for m in stats.get('model_stats', []):
        print(f"  Model: {m['name']} v{m['version']} - state: {m.get('state', 'unknown')}")

## Helper: run perf_analyzer and capture output

In [ ]:
def run_perf_analyzer(model_name, extra_args="", input_data="/workspace/triton/input_data_text.json"):
    """Run perf_analyzer and return output."""
    cmd = (
        f"perf_analyzer -u {TRITON_URL} "
        f"-m {model_name} "
        f"--input-data {input_data} "
        f"{extra_args}"
    )
    print(f"Running: {cmd}")
    result = subprocess.run(
        cmd, shell=True, capture_output=True, text=True, timeout=300
    )
    print(result.stdout)
    if result.stderr:
        print(f"STDERR: {result.stderr}")
    return result.stdout

## 1. Concurrency-Range Mode

Sweep concurrency from 1 to 8 to measure how throughput and latency scale.

In [ ]:
for model_name in MODELS_TO_BENCHMARK:
    print(f"\n{'='*60}")
    print(f"Concurrency sweep: {model_name}")
    print(f"{'='*60}")
    try:
        output = run_perf_analyzer(
            model_name,
            extra_args="--concurrency-range 1:8:1 -b 1"
        )
    except Exception as e:
        print(f"Failed for {model_name}: {e}")

## 2. Request-Rate-Range Mode (Constant Distribution)

Control rate of incoming requests from 10 to 100 req/sec.

In [ ]:
for model_name in MODELS_TO_BENCHMARK:
    print(f"\n{'='*60}")
    print(f"Request-rate sweep (constant): {model_name}")
    print(f"{'='*60}")
    try:
        output = run_perf_analyzer(
            model_name,
            extra_args="--request-rate-range 10:100:10 --request-distribution constant -b 1"
        )
    except Exception as e:
        print(f"Failed for {model_name}: {e}")

## 3. Request-Rate-Range Mode (Poisson Distribution)

Simulate realistic traffic with randomized request arrival times.

In [ ]:
for model_name in MODELS_TO_BENCHMARK:
    print(f"\n{'='*60}")
    print(f"Request-rate sweep (poisson): {model_name}")
    print(f"{'='*60}")
    try:
        output = run_perf_analyzer(
            model_name,
            extra_args="--request-rate-range 10:100:10 --request-distribution poisson -b 1"
        )
    except Exception as e:
        print(f"Failed for {model_name}: {e}")

## 4. ONNX Models with Shape Parameters

For ONNX-backend models (distilbert_classifier, minilm_classifier), use
numeric input data instead of text.

In [ ]:
onnx_models = ["distilbert_classifier", "minilm_classifier", "tfidf_logreg_classifier"]

for model_name in onnx_models:
    print(f"\n{'='*60}")
    print(f"ONNX concurrency sweep: {model_name}")
    print(f"{'='*60}")
    try:
        if "tfidf" in model_name:
            extra = "--concurrency-range 1:8:1 -b 1"
        else:
            extra = (
                "--concurrency-range 1:8:1 -b 1 "
                "--shape input_ids:64 --shape attention_mask:64"
            )
        output = run_perf_analyzer(model_name, extra_args=extra)
    except Exception as e:
        print(f"Failed for {model_name}: {e}")

## 5. Monitor GPU Utilization

Run `nvtop` in a separate terminal during the above benchmarks.

```bash
# On the host (SSH session)
nvtop
```

Or check GPU stats programmatically:

In [ ]:
result = subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True)
print(result.stdout)

## Summary

Results from perf_analyzer are printed inline above. For structured analysis,
parse the CSV output from perf_analyzer:

```bash
perf_analyzer -u triton_server:8000 -m <model> --input-data input.json \
  --concurrency-range 1:8 -f results/triton_<model>_perf.csv
```

In [ ]:
for model_name in MODELS_TO_BENCHMARK:
    csv_path = f"/workspace/evaluation/results/triton_{model_name}_perf.csv"
    try:
        if "tfidf" in model_name or "fasttext" in model_name:
            extra = f"--concurrency-range 1:8:1 -b 1 -f {csv_path}"
        elif "_pt" in model_name:
            extra = f"--concurrency-range 1:8:1 -b 1 -f {csv_path}"
        else:
            extra = (
                f"--concurrency-range 1:8:1 -b 1 "
                f"--shape input_ids:64 --shape attention_mask:64 "
                f"-f {csv_path}"
            )
        run_perf_analyzer(model_name, extra_args=extra,
                         input_data="/workspace/triton/input_data_text.json")
        print(f"Results written to {csv_path}")
    except Exception as e:
        print(f"Failed for {model_name}: {e}")